## 图书域元数据预处理（2023版）
只保留同时有 title 和 images 的图书物品

In [1]:
import json, gzip, csv, os

META_PATH = "./meta_Books.jsonl.gz"
SAVE_DIR = "./processed/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
def pick_best_image(images):
    if not images:
        return ''
    img = images[0]
    for key in ['hi_res', 'large', 'thumb']:
        url = img.get(key, '')
        if url:
            return url
    return ''

total = 0
kept = 0
no_title = 0
no_image = 0

with gzip.open(META_PATH, 'rt', encoding='utf-8') as f_in, \
     open(f'{SAVE_DIR}/book_meta_clean.csv', 'w', encoding='utf-8', newline='') as f_out:
    
    writer = csv.DictWriter(f_out, fieldnames=['parent_asin', 'title', 'description', 'imUrl'])
    writer.writeheader()
    
    for line in f_in:
        total += 1
        try:
            obj = json.loads(line.strip())
        except:
            continue
        
        title = (obj.get('title') or '').strip()
        images = obj.get('images') or []
        
        if not title:
            no_title += 1
            continue
        if not images:
            no_image += 1
            continue
        
        kept += 1
        
        desc = obj.get('description') or []
        if isinstance(desc, list):
            desc = ' '.join(d for d in desc if d and d.strip())
        desc = (desc or '').strip()
        
        writer.writerow({
            'parent_asin': obj.get('parent_asin', ''),
            'title': title,
            'description': desc,
            'imUrl': pick_best_image(images)
        })
        
        if total % 500000 == 0:
            print(f'  已处理 {total/1e6:.1f}M, 保留 {kept} ...')

print(f'总行数: {total}')
print(f'保留: {kept} ({kept/total*100:.1f}%)')
print(f'无title: {no_title}')
print(f'无images: {no_image}')
print(f'输出: {SAVE_DIR}/book_meta_clean.csv')

  已处理 0.5M, 保留 487024 ...
  已处理 1.0M, 保留 974270 ...
  已处理 1.5M, 保留 1325546 ...
  已处理 2.0M, 保留 1733882 ...
  已处理 2.5M, 保留 2221443 ...
  已处理 3.0M, 保留 2708715 ...
总行数: 4448181
保留: 3082232 (69.3%)
无title: 0
无images: 1365949
输出: ./processed//book_meta_clean.csv


In [4]:
import pandas as pd
df = pd.read_csv(f'{SAVE_DIR}/book_meta_clean.csv')
print(f'行数: {len(df)}')
print(f'有描述: {(df["description"]!="").sum()} ({(df["description"]!="").sum()/len(df)*100:.1f}%)')
print(f'\n前5条:')
df.head()

行数: 3082232
有描述: 3082232 (100.0%)

前5条:


,parent_asin,title,description,imUrl
0,0701169850,Chaucer,NaN,https://m.media-amazon.com/images/I/41X61VPJYK...
1,0435088688,Notes from a Kidwatcher,"About the Author SANDRA WILDE, Ph.D., is widel...",https://m.media-amazon.com/images/I/41bfTRxpMM...
2,0316185361,Service: A Navy SEAL at War,"Review Praise for SERVICE""An action-packed...r...",https://m.media-amazon.com/images/I/41YQHDWRyG...
3,0545425573,Monstrous Stories #4: The Day the Mice Stood S...,NaN,https://m.media-amazon.com/images/I/614Mx0QCe7...
4,B00KFOP3RG,Parker & Knight,NaN,https://m.media-amazon.com/images/I/41j6GpAqFB...
